# AeroPulse — Bronze Aircraft Ingestion

## Purpose

Ingest aircraft master data received from the simulated ERP source
system into the Development Bronze layer.

## Source

ERP

## Entity

Aircraft

## Source Format

JSON

## Target

workspace.aeropulse_dev.bronze_aircraft

## Processing Pattern

Raw Landing → Delivery Discovery → Registry → Bronze

## Operational Controls

- Pipeline audit
- Delivery registry
- Rerun safety
- Technical metadata
- Failure tracking


In [0]:
import sys

# Clear cached module to pick up any changes
if 'audit.pipeline_audit' in sys.modules:
    del sys.modules['audit.pipeline_audit']
if 'audit' in sys.modules:
    del sys.modules['audit']

sys.path.append('/Workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src')
from audit.pipeline_audit import start_pipeline_run, complete_pipeline_run, get_utc_timestamp

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/delivery_discovery.py

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/ingestion_registry.py

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/bronze_ingestion.py

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"

SCHEMA = f"aeropulse_{ENVIRONMENT}"

SOURCE_SYSTEM = "erp"

SOURCE_ENTITY = "aircraft"

FILE_FORMAT = "json"

SOURCE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}"
)

BRONZE_TABLE = (
    f"{CATALOG}.{SCHEMA}.bronze_{SOURCE_ENTITY}"
)

AUDIT_TABLE = (
    f"{CATALOG}.{SCHEMA}.audit_pipeline_runs"
)

REGISTRY_TABLE = (
    f"{CATALOG}.{SCHEMA}.ingestion_file_registry"
)

PIPELINE_NAME = (
    f"bronze_{SOURCE_ENTITY}_ingestion"
)

print(f"Source: {SOURCE_PATH}")
print(f"Format: {FILE_FORMAT}")
print(f"Bronze: {BRONZE_TABLE}")
print(f"Registry: {REGISTRY_TABLE}")

In [0]:
# Read from the delivery directory containing the actual JSON files
# The source path has subdirectories for each delivery
aircraft_json_df = (
    spark.read
    .json(f"{SOURCE_PATH}/*")
)

aircraft_json_df.printSchema()

In [0]:
(
    aircraft_json_df
    .limit(0)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print(
    f"Bronze table created: {BRONZE_TABLE}"
)

In [0]:
spark.sql(
    f"DESCRIBE TABLE {BRONZE_TABLE}"
).show(
    truncate=False
)

In [0]:
print(
    spark.table(BRONZE_TABLE).count()
)

In [0]:
from uuid import uuid4

PIPELINE_RUN_ID = str(uuid4())

print(
    f"Pipeline Run ID: {PIPELINE_RUN_ID}"
)


In [0]:
start_pipeline_run(
    spark=spark,
    audit_table=AUDIT_TABLE,
    pipeline_run_id=PIPELINE_RUN_ID,
    pipeline_name=PIPELINE_NAME,
    environment=ENVIRONMENT,
    layer="bronze",
    source_system=SOURCE_SYSTEM,
    target_table=BRONZE_TABLE,
)

In [0]:
metrics = ingest_bronze_deliveries(
    spark=spark,
    dbutils=dbutils,
    source_path=SOURCE_PATH,
    source_entity=SOURCE_ENTITY,
    source_system=SOURCE_SYSTEM,
    file_format=FILE_FORMAT,
    bronze_table=BRONZE_TABLE,
    registry_table=REGISTRY_TABLE,
    pipeline_run_id=PIPELINE_RUN_ID,
)


In [0]:
complete_pipeline_run(
    spark=spark,
    audit_table=AUDIT_TABLE,
    pipeline_run_id=PIPELINE_RUN_ID,
    pipeline_status="SUCCESS",
    records_read=metrics["records_read"],
    records_inserted=metrics["records_inserted"],
    records_updated=0,
    records_rejected=0,
)


In [0]:
aircraft_bronze_count = (
    spark.table(BRONZE_TABLE)
    .count()
)

print(
    f"Bronze aircraft records: "
    f"{aircraft_bronze_count}"
)

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_TABLE}
        LIMIT 20
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            airline_id,
            COUNT(*) AS aircraft_count
        FROM {BRONZE_TABLE}
        GROUP BY airline_id
        ORDER BY airline_id
    """)
)


In [0]:
display(
    spark.sql(f"""
        SELECT
            source_file_name,
            source_entity,
            file_format,
            ingestion_status,
            records_read,
            records_inserted,
            pipeline_run_id
        FROM {REGISTRY_TABLE}
        WHERE source_entity = 'aircraft'
        ORDER BY ingestion_start_timestamp DESC
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            pipeline_name,
            pipeline_status,
            records_read,
            records_inserted,
            start_timestamp,
            end_timestamp,
            error_message
        FROM {AUDIT_TABLE}
        WHERE pipeline_run_id = '{PIPELINE_RUN_ID}'
    """)
)

In [0]:
PIPELINE_RUN_ID = str(uuid4())

In [0]:
metrics = ingest_bronze_deliveries(
    spark=spark,
    dbutils=dbutils,
    source_path=SOURCE_PATH,
    source_entity=SOURCE_ENTITY,
    source_system=SOURCE_SYSTEM,
    file_format=FILE_FORMAT,
    bronze_table=BRONZE_TABLE,
    registry_table=REGISTRY_TABLE,
    pipeline_run_id=PIPELINE_RUN_ID,
)

In [0]:
spark.table(BRONZE_TABLE).count()
